In [1]:
import pandas as pd
import faiss
import joblib
from sentence_transformers import SentenceTransformer

CSV_PATH = './Data/dataset.csv'
EMBEDDING_MODEL = 'keepitreal/vietnamese-sbert'

simu = 0.8

K:\Deeplearning\ChatbotAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

from Lmd_Utils import cleantext
df = pd.read_parquet('./Data/train.parquet')
df2 = pd.read_parquet('./Data/validation.parquet')
df = pd.concat([df, df2], ignore_index=True)
questions, answers = [], []


df = df[['question', 'context']].dropna()

questions = df['question'].tolist()
answers = df['context'].tolist()



[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lemin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\lemin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 32928.30it/s]
RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3
0
2
5051
1
Tôi không tìm thấy thông tin xác định về tên gọi trước khi Phạm Văn Đồng làm Phó chủ nhiệm cơ quiang Biện sự xứ tại Quế Lâm. Thông tin tham khảo chủ yếu đề cập đến Lâm Bá Kiệt là Thủ tướng Việt Nam Dân chủ Cộng hòa năm 1954, chứ không liên quan đến Phạm Văn Đồng.


In [3]:

embed_model = SentenceTransformer(EMBEDDING_MODEL)

question_embeddings = embed_model.encode(
    questions,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f'✅ Embedding shape: {question_embeddings.shape}')


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11785.91it/s]
RobertaModel LOAD REPORT from: keepitreal/vietnamese-sbert
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 505/505 [07:51<00:00,  1.07it/s]

✅ Embedding shape: (32268, 768)


In [4]:

faiss.normalize_L2(question_embeddings)
n = question_embeddings.shape[1]
index = faiss.IndexFlatIP(n)
index.add(question_embeddings)


faiss.write_index(index, './models_rag/faiss_index.bin')
joblib.dump(answers, './models_rag/answers.pkl')
joblib.dump(questions, './models_rag/questions.pkl')


['./models_rag/questions.pkl']